# Global Coffee & Living Index 2026: End-to-End Pipeline & EDA

## Project Summary
This notebook presents an end-to-end exploratory analysis of the **Global Coffee & Living Index 2026** dataset pipeline. The dataset combines macroeconomic data from the **World Bank**, trade and production statistics from **FAOSTAT (UN FAO)**, and geographical capital information.

### Structure of the Notebook
1. **Environment Setup & Data Ingestion**: Automated path resolution and dataset loading.
2. **Raw Dataset Audit**:
   - `worldbank_indicators_raw.csv`: Gross National Income (GNI), Price Level Index (PLI), and Purchasing Power Parity (PPP).
   - `faostat_coffee_raw.csv`: Green coffee production quantity and export values.
   - `country_capitals_raw.csv`: Capital and representative cities by ISO3 code.
3. **Consolidated Dataset Audit**:
   - `global_coffee_living_index.csv`: Integrated cross-country index with computed purchasing power metrics.
4. **Exploratory Data Analysis (EDA) & Visualizations**: Cost of living vs. income distribution and affordability scores.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Locate data files dynamically within the Kaggle input environment or local folder
data_files = {}
target_files = [
    'worldbank_indicators_raw.csv',
    'faostat_coffee_raw.csv',
    'country_capitals_raw.csv',
    'global_coffee_living_index.csv'
]

# Search directory tree
for root, dirs, files in os.walk('.'):
    for file in files:
        if file in target_files and file not in data_files:
            data_files[file] = os.path.join(root, file)

# Search Kaggle input directory if available
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file in target_files:
                data_files[file] = os.path.join(root, file)

print("Detected Data Files:")
for name, path in data_files.items():
    print(f" - {name}: {path}")

## Section 1: World Bank Macroeconomic Indicators (`worldbank_indicators_raw.csv`)

The World Bank dataset serves as the foundational economic baseline for the project.

### Metrics Included:
* **GNI per Capita (USD)**: Gross National Income converted to US dollars using the Atlas method (`NY.GNP.PCAP.CD`).
* **Price Level Index (PLI)**: Ratio of a country's PPP conversion factor to its market exchange rate (`PA.NUS.PRCE.GG`).
* **PPP Conversion Factor**: Local currency units per international dollar (`PA.NUS.PPPC.RF`).

In [ ]:
wb_df = pd.read_csv(data_files['worldbank_indicators_raw.csv'])

print(f"Shape of World Bank Data: {wb_df.shape[0]} rows, {wb_df.shape[1]} columns\n")
print("Data Types & Null Counts:")
print(wb_df.info())

print("\nSummary Statistics:")
display(wb_df.describe().T)

print("\nSample Records (First 5 Rows):")
display(wb_df.head(5))

## Section 2: FAOSTAT Coffee Data (`faostat_coffee_raw.csv`)

This raw dataset extracts national coffee footprint figures directly from the Food and Agriculture Organization (FAOSTAT).

### Metrics Included:
* **Coffee Production Quantity (`coffee_production_qty`)**: Measured in metric tonnes (Domain `QCL`, Element `5510`).
* **Coffee Export Value (`coffee_export_value_usd1000`)**: Total export value measured in thousand USD (Domain `TCL`, Element `5922`).

In [ ]:
fao_df = pd.read_csv(data_files['faostat_coffee_raw.csv'])

print(f"Shape of FAOSTAT Data: {fao_df.shape[0]} rows, {fao_df.shape[1]} columns\n")
print("Columns Present:", list(fao_df.columns))

if fao_df.empty:
    print("\nNote: FAOSTAT raw dataset currently contains structural headers only due to API offline fallback.")
else:
    print("\nNon-null Records Count:")
    print(fao_df.notnull().sum())
    display(fao_df.head(5))

## Section 3: Country Capitals (`country_capitals_raw.csv`)

Provides geographical mapping of representative cities and capitals matched against standardized ISO 3166-1 alpha-3 country codes (`iso3_code`).

In [ ]:
capitals_df = pd.read_csv(data_files['country_capitals_raw.csv'])

print(f"Shape of Country Capitals Data: {capitals_df.shape[0]} rows, {capitals_df.shape[1]} columns\n")
print("Missing Value Summary:")
print(capitals_df.isnull().sum())

print("\nSample Capital Mappings:")
display(capitals_df.dropna().head(10))

## Section 4: Final Consolidated Dataset (`global_coffee_living_index.csv`)

The primary dataset produced by merging all three raw sources on `iso3_code` and computing custom relative affordability metrics.

### Key Computed Fields:
1. **Purchasing Power Ratio**:
   $$\text{Purchasing Power Ratio} = \frac{\text{GNI per Capita (USD)}}{\text{Price Level Index}}$$
2. **Living Affordability Score**:
   $$\text{Score} = \left( \frac{\text{PPR} - \text{PPR}_{\min}}{\text{PPR}_{\max} - \text{PPR}_{\min}} \right) \times 100$$

In [ ]:
main_df = pd.read_csv(data_files['global_coffee_living_index.csv'])

print(f"Consolidated Dataset Shape: {main_df.shape[0]} countries/territories, {main_df.shape[1]} attributes\n")

# Complete Data Audit Table
audit_df = pd.DataFrame({
    'Data Type': main_df.dtypes,
    'Non-Null Count': main_df.notnull().sum(),
    'Null Count': main_df.isnull().sum(),
    'Null Percentage (%)': (main_df.isnull().sum() / len(main_df) * 100).round(2)
})

print("Dataset Audit & Missing Value Summary:")
display(audit_df)

## Section 5: Exploratory Data Analysis & Visualizations

Visualizing the distribution of living affordability across global economies and examining the relationship between local price levels and gross income.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Visualization 1: Top 10 Countries by Living Affordability Score
top10 = main_df.dropna(subset=['living_affordability_score']).sort_values(
    by='living_affordability_score', ascending=False
).head(10)

sns.barplot(
    data=top10,
    x='living_affordability_score',
    y='country_name',
    hue='country_name',
    legend=False,
    palette='crest',
    ax=axes[0]
)
axes[0].set_title('Top 10 Economies by Living Affordability Score')
axes[0].set_xlabel('Affordability Score (0 - 100)')
axes[0].set_ylabel('Country')

# Visualization 2: GNI per Capita vs Price Level Index
sns.scatterplot(
    data=main_df,
    x='price_level_index',
    y='gni_per_capita_usd',
    hue='living_affordability_score',
    palette='viridis',
    size='living_affordability_score',
    sizes=(30, 250),
    ax=axes[1]
)
axes[1].set_title('GNI per Capita vs. Price Level Index (PLI)')
axes[1].set_xlabel('Price Level Index (US = 100 Baseline)')
axes[1].set_ylabel('GNI per Capita (USD)')
axes[1].set_yscale('log') # Log scale for income clarity

plt.tight_layout()
plt.show()